# Farming Score V4


## 1. Failure mechanism

Let $b\in\{\mathrm{COW},\mathrm{SHEEP}\}$ be the animal selected for the registered turn-$313$ bundle. The original latent-pasture rule purchases two copies,

$$
\Delta n = 2e_b,
$$

so one threshold decision receives gain two. In six supplied defeats, the source sheep was first relabelled as a cow and then duplicated, producing a final $10/4$ cow-sheep mix. Most winning opponents preserved at least five sheep.


## 2. Supply-adjusted conversion

For visible opponent counts $C_o$ and $S_o$, define

$$
B_o=\frac{C_o-S_o}{\max(1,C_o+S_o)}.
$$

Only the cow-to-sheep decision is adjusted:

$$
G_{C\rightarrow S}
=P_{\mathrm{wool}}-P_{\mathrm{dairy}}+B_o.
$$

A cow bundle is relabelled when $G_{C\rightarrow S}\ge1$. A cow-heavy opponent makes wool relatively less supplied. This activates the existing quantity-two conversion in the Yarn-heavy defeat while leaving all six non-Yarn openings unchanged.


## 3. Diversified latent pasture

When the late source sheep has already been relabelled as a cow, the extra pasture receives the complementary sheep:

$$
2e_{\mathrm{COW}}
\quad\longrightarrow\quad
e_{\mathrm{COW}}+e_{\mathrm{SHEEP}}.
$$

Under a strong wool signal, the original two-sheep bundle remains unchanged. Purchase funding, hand spawn geometry, inventory, movement, and placement are checked from the current observation; any mismatch returns control to the parent action.


## 4. Conserved structure

Both interventions operate only on pre-registered pasture-compatible transactions. They preserve

$$
\begin{aligned}
\text{pasture geometry}' &= \text{pasture geometry},\\
\text{animal count}' &= \text{animal count},\\
\text{feed schedule}' &= \text{feed schedule},\\
\text{worker count}' &= \text{worker count}.
\end{aligned}
$$

The complete final-turn frontier is retained: existing sales are expanded without a new slot, and scarce new slots maximize exact positive terminal revenue.


In [ ]:
# Decode, verify, and write the complete submission package.
import base64
import hashlib
import io
import json
from pathlib import Path
import tarfile

ARCHIVE_B64 = (
    "..."
    "478z/m/Zc+JMALgBAA=="
)
EXPECTED_ARCHIVE_SHA256 = "221a717a69a3f51485e2f7476612caaa94b1bc29bd2c6a26aef4f07ee9f102c3"
EXPECTED_MEMBER_HASHES = json.loads('{"agents/complete_terminal_frontier.py": "1a892491b01b1e25d22db3a3f77e1cceed6f56190c0b4d4f9565e2ef7e92e576", "agents/e749a_niklita_consensus_network.py": "2188debac2af3308f4ea1bd427a38cd3a2332394073c7cca6240c011fb0c5374", "agents/e750a_place_funding_repair.py": "b0e12951d73646f37eb3b67531716237520f67cdea723914a87bda764c712dc0", "agents/e766a_universal_kenjo_medoid.py": "b66a4acb25695ce7e04b4a9a57c60543926fdc3bd2e85fc827c82bc80d86dc45", "agents/e773a_demand_aligned_pasture_network.py": "d7c9aa8cd13cc01e3ce61e7e14bef9880f4aae8cfa476d45d6ce97acb9e8adec", "agents/e774a_terminal_animal_frontier.py": "bc446060cb4ca57e31cc7583f0da16e92c31cb0818d84435483f7523282f4b3c", "agents/e775a_latent_pasture_activation.py": "4abb721b60f8928683ce038903f616cc618ce9a547c7a1f520b1fb81a7398bc6", "agents/e776a_engine_exact_latent_pasture.py": "2c603fd6f9bf978d7e8eef78a34647bd379d0e89963ada0cbb1c52c39b0f4b3f", "agents/late_bundle_diversifier.py": "1951a759a6f0cff8b113733a21daf796eab3daa2228cd61249d8d6d85da83508", "artifacts/e706_top10_tapes/episode_101408728_seat1.py": "685cdf4d9b14f16985e0dc7bdc0770446c3862a233be5dfe1f0efe508eedb629", "artifacts/e751_current_top10_tapes/episode_102192548_seat1.py": "4d686ff547797f776081254ee602eda151ec0fe5de7ddcdf646fedf1bf5f0f89", "configs/server_environment_20260807.json": "ca5cb5d68d2b3cca65016c6b4eb62d3a5aa1c463720b7c2fe7bc5689a38e5d1a", "e776_pkg/__init__.py": "b001950885c35bcf24367510245f41667e85bad8d754072c3489afdcf97184c4", "e776_pkg/entry.py": "5aecede545c47e7b2ea19228bd1bd9d0083983e65d4f8d01a940a6470a5ef4ad", "main.py": "e8498c67914ecc607ae69fde25a728361eb5acea94c00ecc85deffbdafe50413", "optimized_pkg/__init__.py": "f1e043b916ff50e95d071dc36888ec2a6cda0e9f460d2d34dabbdaa9f033854a", "optimized_pkg/entry.py": "d8e0cb1200f024e88a6103907ca66e829036694b05d1d56ffa5a7b65f247a81a"}')
ARCHIVE_BYTES = base64.b64decode(ARCHIVE_B64)
assert hashlib.sha256(ARCHIVE_BYTES).hexdigest() == EXPECTED_ARCHIVE_SHA256
with tarfile.open(fileobj=io.BytesIO(ARCHIVE_BYTES), mode="r:gz") as package:
    members = {member.name: package.extractfile(member).read()
               for member in package.getmembers() if member.isfile()}
assert set(members) == set(EXPECTED_MEMBER_HASHES)
for name, data in members.items():
    assert hashlib.sha256(data).hexdigest() == EXPECTED_MEMBER_HASHES[name]

ARCHIVE_PATH = Path.cwd() / "submission.tar.gz"
MAIN_PATH = Path.cwd() / "main.py"
ARCHIVE_PATH.write_bytes(ARCHIVE_BYTES)
MAIN_PATH.write_bytes(members["main.py"])
print("archive:", ARCHIVE_PATH)
print("members:", len(members))


In [ ]:
# Extract and import the packaged entry point.
import importlib.util
import sys

PACKAGE_DIR = Path.cwd() / "generated_submission"
PACKAGE_DIR.mkdir(exist_ok=True)
with tarfile.open(ARCHIVE_PATH, "r:gz") as package:
    package.extractall(PACKAGE_DIR)
sys.path.insert(0, str(PACKAGE_DIR))
spec = importlib.util.spec_from_file_location(
    "generated_submission_main", PACKAGE_DIR / "main.py"
)
submission = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = submission
spec.loader.exec_module(submission)
assert callable(submission.kaggriculture_agent)
print("standalone package import: OK")


## Attribution

The complete programme priors are the attributed CC0 public traces from Kenjo1209, episode 102192548 seat 1, and NIklitaCheporev, episode 101408728 seat 1. The supplied execution network is retained; the added contribution is the public-supply adjustment, diversified latent bundle, and complete terminal frontier.
